# Differential Evolution for the 2D Rastrigin Function

Comprehensive experimental study of Differential Evolution (DE) variants and parameter sensitivity.

**Problem:** Minimize the 2D Rastrigin function
**Population size:** 10 (5x dimension)
**Generations:** 500
**Total experiments:** 55 runs across 5 experiment types

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time, os, json

# Configure plotting
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f8'

# Output directory for results
OUTDIR = './DE_Results'
os.makedirs(OUTDIR, exist_ok=True)

print(f'Output directory: {OUTDIR}')

Output directory: ./DE_Results


## Problem Definition

**Rastrigin Function:**
$$f(\mathbf{x}) = 20 + \sum_{i=1}^{n} \left( x_i^2 - 10 \cos(2\pi x_i) \right)$$

- **Domain:** $[-5.12, 5.12]^2$
- **Global minimum:** $\mathbf{x}^* = (0, 0)$ with $f(\mathbf{x}^*) = 0$
- **Characteristics:** Highly multimodal with many local minima

**Optimization Parameters:**
- Population Size (ps): 10
- Max Generations: 500
- Strategies: rand, best
- F values (Mutation scale): [0.1, 0.3, 0.5, 0.8, 1.2]
- CR values (Crossover rate): [0.0, 0.2, 0.5, 0.7, 0.9]
- Seeds: [1, 2, 3, 4, 5]

In [2]:
# Problem constants
LOWER, UPPER = -5.12, 5.12
N_VARS = 2
PS = 10
MAX_GEN = 500

def rastrigin(x):
    """Rastrigin benchmark function."""
    x = np.asarray(x)
    if x.ndim == 1:
        return 20 + (x[0]**2 - 10*np.cos(2*np.pi*x[0])) + (x[1]**2 - 10*np.cos(2*np.pi*x[1]))
    else:
        return 20 + np.sum(x**2 - 10*np.cos(2*np.pi*x), axis=1)

# Test the function
test_point = np.array([0, 0])
print(f'Rastrigin at optimum (0,0): {rastrigin(test_point):.10f}')
print(f'Rastrigin at (1,1): {rastrigin(np.array([1, 1])):.6f}')

Rastrigin at optimum (0,0): 0.0000000000
Rastrigin at (1,1): 2.000000


## Rastrigin Function Visualization

In [3]:
print('Generating surface plot...')
x1 = np.linspace(LOWER, UPPER, 200)
x2 = np.linspace(LOWER, UPPER, 200)
X1, X2 = np.meshgrid(x1, x2)
Z = 20 + (X1**2 - 10*np.cos(2*np.pi*X1)) + (X2**2 - 10*np.cos(2*np.pi*X2))

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X1, X2, Z, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.set_xlabel('x1')
ax1.set_ylabel('x2')
ax1.set_zlabel('f(x)')
ax1.set_title('Rastrigin Function - 3D Surface', fontsize=12, fontweight='bold')

ax2 = fig.add_subplot(122)
c = ax2.contourf(X1, X2, Z, levels=50, cmap='viridis')
plt.colorbar(c, ax=ax2)
ax2.plot(0, 0, 'r*', markersize=15, label='Global min (0,0)')
ax2.set_xlabel('x1')
ax2.set_ylabel('x2')
ax2.set_title('Rastrigin Function - Contour', fontsize=12, fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.savefig(f'{OUTDIR}/rastrigin_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Surface plot saved.')

Generating surface plot...
Surface plot saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/2549871697.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Differential Evolution Algorithm

**Strategies:**
- **DE/rand/1/bin:** Base individual selected randomly
- **DE/best/1/bin:** Base individual is the best in population

**Mutation (Differential Mutation):**
$$\mathbf{v}_i = \mathbf{x}_{base} + F (\mathbf{x}_{r1} - \mathbf{x}_{r2})$$

**Crossover (Binomial):**
$$u_{j,i} = \begin{cases} v_{j,i} & \text{if } \text{rand}() < CR \text{ or } j = j_{rand} \\ x_{j,i} & \text{otherwise} \end{cases}$$

**Selection:**
$$\mathbf{x}_{i}^{t+1} = \begin{cases} \mathbf{u}_i & \text{if } f(\mathbf{u}_i) \le f(\mathbf{x}_i^t) \\ \mathbf{x}_i^t & \text{otherwise} \end{cases}$$

In [4]:
def run_de(ps=PS, F=0.8, CR=0.9, max_gen=MAX_GEN, seed=42,
           strategy='rand', dynamic_F=False, F_low=0.5, F_high=1.0):
    """
    Run Differential Evolution optimizer.
    
    Parameters:
    -----------
    ps : int
        Population size
    F : float
        Mutation scale factor (0 < F <= 2)
    CR : float
        Crossover probability (0 <= CR <= 1)
    max_gen : int
        Maximum number of generations
    seed : int
        Random seed for reproducibility
    strategy : str
        'rand' or 'best'
    dynamic_F : bool
        If True, F is sampled from U(F_low, F_high) each generation
    F_low, F_high : float
        Bounds for dynamic F sampling
    
    Returns:
    --------
    best_x : ndarray
        Best solution found
    best_cost : float
        Cost of best solution
    history : list
        Best cost at each generation
    pop : ndarray
        Final population
    fitness : ndarray
        Final fitness values
    """
    rng = np.random.RandomState(seed)
    n = N_VARS
    
    # Initialize population
    pop = rng.uniform(LOWER, UPPER, size=(ps, n))
    fitness = rastrigin(pop)
    best_idx = np.argmin(fitness)
    best_x = pop[best_idx].copy()
    best_cost = fitness[best_idx]
    history = [best_cost]
    
    # Main loop
    for gen in range(max_gen):
        F_gen = rng.uniform(F_low, F_high) if dynamic_F else F
        
        for i in range(ps):
            # Select mutation indices
            candidates = list(range(ps))
            candidates.remove(i)
            
            if strategy == 'best':
                base_idx = best_idx
                remaining = [c for c in candidates if c != best_idx]
                if len(remaining) < 2:
                    remaining = candidates
                idxs = rng.choice(remaining, 2, replace=False)
                r1, r2 = idxs[0], idxs[1]
            else:  # strategy == 'rand'
                idxs = rng.choice(candidates, 3, replace=False)
                base_idx, r1, r2 = idxs[0], idxs[1], idxs[2]
            
            # Mutation
            donor = pop[base_idx] + F_gen * (pop[r1] - pop[r2])
            
            # Crossover (binomial)
            trial = pop[i].copy()
            j_rand = rng.randint(n)
            for j in range(n):
                if rng.random() < CR or j == j_rand:
                    trial[j] = donor[j]
            
            # Boundary handling (reinitialize if out of bounds)
            for j in range(n):
                if trial[j] < LOWER or trial[j] > UPPER:
                    trial[j] = rng.uniform(LOWER, UPPER)
            
            # Selection
            trial_cost = rastrigin(trial)
            if trial_cost <= fitness[i]:
                pop[i] = trial
                fitness[i] = trial_cost
                if trial_cost < best_cost:
                    best_cost = trial_cost
                    best_x = trial.copy()
                    best_idx = i
        
        history.append(best_cost)
    
    return best_x, best_cost, history, pop, fitness

print('DE algorithm defined.')

DE algorithm defined.


## Utility Functions for Safe Logging and Plotting

In [5]:
def safe_log(vals):
    """Safely convert values to log scale, avoiding log(0)."""
    return [max(v, 1e-16) for v in vals]

def safe_log_hist(hist):
    """Safely convert history to log scale."""
    return [max(v, 1e-16) for v in hist]

print('Helper functions defined.')

Helper functions defined.


## Initial Population Analysis

Analyze the diversity and fitness distribution of random initial populations.

In [6]:
print('Generating initial population boxplot...')
fig, ax = plt.subplots(figsize=(8, 5))
init_data = []
for seed in [1, 2, 3, 4, 5]:
    rng = np.random.RandomState(seed)
    pop = rng.uniform(LOWER, UPPER, size=(PS, N_VARS))
    fits = rastrigin(pop)
    init_data.append(fits)

bp = ax.boxplot(init_data, tick_labels=[f'Seed {s}' for s in [1, 2, 3, 4, 5]], patch_artist=True)
colors_seed = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12']
for patch, color in zip(bp['boxes'], colors_seed):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Objective Function Value', fontsize=12, fontweight='bold')
ax.set_title('Objective Function Values for Initial Populations', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', linestyle='-.', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/initial_populations.png', dpi=300, bbox_inches='tight')
plt.show()

all_init = np.concatenate(init_data)
print(f'Initial populations: grand mean = {np.mean(all_init):.2f}, std = {np.std(all_init):.2f}')

Generating initial population boxplot...
Initial populations: grand mean = 33.58, std = 14.61


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/3167057566.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 1: DE/rand/1/bin

**Base vector:** randomly selected individual
**Mutation scale:** F = 0.8
**Crossover rate:** CR = 0.9
**Seeds:** [1, 2, 3, 4, 5]

In [7]:
print('='*65)
print(f'EXPERIMENT 1: DE/rand/1/bin — F=0.8, CR=0.9, ps={PS}, max_gen={MAX_GEN}')
print('='*65)

seeds = [1, 2, 3, 4, 5]
results_all = []
run_id = 0

for seed in seeds:
    run_id += 1
    t0 = time.perf_counter()
    bx, bc, hist, fpop, ffit = run_de(ps=PS, F=0.8, CR=0.9, max_gen=MAX_GEN, seed=seed, strategy='rand')
    runtime = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'Exp1_rand',
        'seed': seed, 'strategy': 'rand', 'F': 0.8, 'CR': 0.9, 'ps': PS, 'dynamic_F': False,
        'best_x1': bx[0], 'best_x2': bx[1], 'best_cost': bc,
        'generations': len(hist), 'history': hist, 'runtime_s': runtime,
        'final_pop_mean': float(np.mean(ffit)), 'final_pop_std': float(np.std(ffit))
    })
    print(f'  Seed {seed}: f = {bc:.8f}  at ({bx[0]:.6f}, {bx[1]:.6f})  [{runtime:.3f}s]  pop_std={np.std(ffit):.6f}')

exp1_costs = [r['best_cost'] for r in results_all if r['experiment'] == 'Exp1_rand']
print(f'\nExp1 summary: mean={np.mean(exp1_costs):.8f}, std={np.std(exp1_costs):.8f}')

EXPERIMENT 1: DE/rand/1/bin — F=0.8, CR=0.9, ps=10, max_gen=500
  Seed 1: f = 0.99495906  at (0.994959, 0.000000)  [0.056s]  pop_std=0.000000
  Seed 2: f = 0.00000000  at (-0.000000, 0.000000)  [0.040s]  pop_std=0.000000
  Seed 3: f = 0.99495906  at (-0.000000, 0.994959)  [0.035s]  pop_std=0.000000
  Seed 4: f = 0.00000000  at (0.000000, 0.000000)  [0.035s]  pop_std=0.000000
  Seed 5: f = 0.99495906  at (0.000000, 0.994959)  [0.035s]  pop_std=0.000000

Exp1 summary: mean=0.59697543, std=0.48742840


## Experiment 2: DE/best/1/bin

**Base vector:** best individual in population
**Mutation scale:** F = 0.8
**Crossover rate:** CR = 0.9
**Seeds:** [1, 2, 3, 4, 5]

In [8]:
print('\n' + '='*65)
print(f'EXPERIMENT 2: DE/best/1/bin — F=0.8, CR=0.9, ps={PS}, max_gen={MAX_GEN}')
print('='*65)

for seed in seeds:
    run_id += 1
    t0 = time.perf_counter()
    bx, bc, hist, fpop, ffit = run_de(ps=PS, F=0.8, CR=0.9, max_gen=MAX_GEN, seed=seed, strategy='best')
    runtime = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'Exp2_best',
        'seed': seed, 'strategy': 'best', 'F': 0.8, 'CR': 0.9, 'ps': PS, 'dynamic_F': False,
        'best_x1': bx[0], 'best_x2': bx[1], 'best_cost': bc,
        'generations': len(hist), 'history': hist, 'runtime_s': runtime,
        'final_pop_mean': float(np.mean(ffit)), 'final_pop_std': float(np.std(ffit))
    })
    print(f'  Seed {seed}: f = {bc:.8f}  at ({bx[0]:.6f}, {bx[1]:.6f})  [{runtime:.3f}s]  pop_std={np.std(ffit):.6f}')

exp2_costs = [r['best_cost'] for r in results_all if r['experiment'] == 'Exp2_best']
print(f'\nExp2 summary: mean={np.mean(exp2_costs):.8f}, std={np.std(exp2_costs):.8f}')

# Determine best strategy
if np.mean(exp1_costs) <= np.mean(exp2_costs):
    best_strategy_12 = 'rand'
    print(f'\n>>> Best strategy: DE/rand/1/bin (mean={np.mean(exp1_costs):.8f})')
else:
    best_strategy_12 = 'best'
    print(f'\n>>> Best strategy: DE/best/1/bin (mean={np.mean(exp2_costs):.8f})')


EXPERIMENT 2: DE/best/1/bin — F=0.8, CR=0.9, ps=10, max_gen=500
  Seed 1: f = 2.19019741  at (-1.026759, 0.994959)  [0.037s]  pop_std=0.000000
  Seed 2: f = 0.05351581  at (0.005622, -0.015438)  [0.036s]  pop_std=0.000000
  Seed 3: f = 2.68502014  at (1.052857, -1.008267)  [0.035s]  pop_std=0.000000
  Seed 4: f = 4.97479025  at (1.989912, -0.994959)  [0.036s]  pop_std=0.000000
  Seed 5: f = 1.18644192  at (-0.000000, -1.026051)  [0.036s]  pop_std=0.000000

Exp2 summary: mean=2.21799311, std=1.64729336

>>> Best strategy: DE/rand/1/bin (mean=0.59697543)


## Comparison: Exp1 vs Exp2

Convergence curves and cost distributions for both strategies.

In [9]:
print('Generating Exp1 vs Exp2 comparison plots...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Convergence comparison
ax = axes[0]
rand_hists = [r['history'] for r in results_all if r['experiment']=='Exp1_rand']
best_hists = [r['history'] for r in results_all if r['experiment']=='Exp2_best']
min_len = min(min(len(h) for h in rand_hists), min(len(h) for h in best_hists))
avg_rand = np.mean([h[:min_len] for h in rand_hists], axis=0)
avg_best = np.mean([h[:min_len] for h in best_hists], axis=0)

for r in results_all:
    if r['experiment']=='Exp1_rand':
        ax.plot(safe_log_hist(r['history']), alpha=0.3, color='steelblue', linewidth=0.8)
    elif r['experiment']=='Exp2_best':
        ax.plot(safe_log_hist(r['history']), alpha=0.3, color='coral', linewidth=0.8)

ax.plot(safe_log_hist(list(avg_rand)), color='navy', linewidth=2.5, label='rand (avg)')
ax.plot(safe_log_hist(list(avg_best)), color='darkred', linewidth=2.5, label='best (avg)')
ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Convergence: rand vs best', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='-.', alpha=0.5)

# Right: Cost distribution
ax = axes[1]
bp = ax.boxplot([safe_log(exp1_costs), safe_log(exp2_costs)],
                tick_labels=['DE/rand/1/bin', 'DE/best/1/bin'], patch_artist=True)
bp['boxes'][0].set_facecolor('#3498db')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#e74c3c')
bp['boxes'][1].set_alpha(0.7)
ax.set_yscale('log')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Cost Distribution: rand vs best', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', linestyle='-.', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp1_vs_exp2_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Exp1 vs Exp2 comparison saved.')

Generating Exp1 vs Exp2 comparison plots...
Exp1 vs Exp2 comparison saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/1648128723.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
fig, ax = plt.subplots(figsize=(10, 5))

for r in results_all:
    if r['experiment']=='Exp1_rand':
        ax.plot(safe_log_hist(r['history']), alpha=0.6, linewidth=1.2, color='steelblue', 
                label=f"rand s{r['seed']}" if r['seed']==1 else None)
    elif r['experiment']=='Exp2_best':
        ax.plot(safe_log_hist(r['history']), alpha=0.6, linewidth=1.2, color='coral', 
                label=f"best s{r['seed']}" if r['seed']==1 else None)

ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Convergence per Seed: rand vs best', fontsize=14, fontweight='bold')
ax.legend(ncol=2)
ax.grid(True, linestyle='-.', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp1_vs_exp2_per_seed.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/1315916675.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 3: Mutation Scale (F) Sweep

**Objective:** Determine optimal F value
**F values tested:** [0.1, 0.3, 0.5, 1.2] (plus baseline 0.8)
**Strategy:** Best strategy from Exp1/Exp2 comparison
**Runs per F:** 5 seeds
**CR:** Fixed at 0.9

In [11]:
F_values = [0.1, 0.3, 0.5, 1.2]

print('\n' + '='*65)
print(f'EXPERIMENT 3: Varying F — strategy={best_strategy_12}, CR=0.9, ps={PS}')
print('='*65)

for F_val in F_values:
    print(f'\n  F = {F_val}')
    for seed in seeds:
        run_id += 1
        t0 = time.perf_counter()
        bx, bc, hist, fpop, ffit = run_de(ps=PS, F=F_val, CR=0.9, max_gen=MAX_GEN, seed=seed, strategy=best_strategy_12)
        runtime = time.perf_counter() - t0
        results_all.append({
            'run_id': run_id, 'experiment': 'Exp3_F_sweep',
            'seed': seed, 'strategy': best_strategy_12, 'F': F_val, 'CR': 0.9, 'ps': PS, 'dynamic_F': False,
            'best_x1': bx[0], 'best_x2': bx[1], 'best_cost': bc,
            'generations': len(hist), 'history': hist, 'runtime_s': runtime,
            'final_pop_mean': float(np.mean(ffit)), 'final_pop_std': float(np.std(ffit))
        })
        print(f'    Seed {seed}: f = {bc:.8f}  at ({bx[0]:.6f}, {bx[1]:.6f})')

# Summary
all_F_vals = F_values + [0.8]
best_F_cost = float('inf')
best_F_val = 0.8

print('\n--- F summary ---')
for fv in sorted(all_F_vals):
    if fv == 0.8:
        costs = [r['best_cost'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
    else:
        costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==fv]
    m = np.mean(costs)
    print(f'  F={fv}: mean={m:.6f}, std={np.std(costs):.6f}')
    if m < best_F_cost:
        best_F_cost = m
        best_F_val = fv

print(f'\n>>> Best F: {best_F_val} (mean={best_F_cost:.8f})')


EXPERIMENT 3: Varying F — strategy=rand, CR=0.9, ps=10

  F = 0.1
    Seed 1: f = 1.33278120  at (-1.036246, 0.001962)
    Seed 2: f = 1.81478494  at (-1.006207, 0.063714)
    Seed 3: f = 2.89910891  at (1.076733, 0.055263)
    Seed 4: f = 8.47141909  at (1.939278, -1.982204)
    Seed 5: f = 5.95130064  at (-0.117408, 1.105448)

  F = 0.3
    Seed 1: f = 0.00038474  at (0.000327, 0.001354)
    Seed 2: f = 0.06978725  at (-0.001337, 0.018718)
    Seed 3: f = 0.74868259  at (0.041392, 0.045653)
    Seed 4: f = 1.13184289  at (-0.995048, -0.026297)
    Seed 5: f = 2.37342282  at (-0.965004, 1.027246)

  F = 0.5
    Seed 1: f = 0.28223818  at (0.026263, -0.027133)
    Seed 2: f = 0.00001885  at (-0.000000, 0.000308)
    Seed 3: f = 0.00446351  at (-0.004167, 0.002265)
    Seed 4: f = 0.99495906  at (-0.994959, 0.000000)
    Seed 5: f = 0.12207309  at (-0.024824, 0.000569)

  F = 1.2
    Seed 1: f = 0.99495906  at (-0.994959, -0.000000)
    Seed 2: f = 0.39185979  at (0.029203, 0.033599)
 

## F Sweep Results Visualization

In [12]:
print('Generating F sweep plots...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence by F
ax = axes[0]
colors_F = {0.1:'#e74c3c', 0.3:'#f39c12', 0.5:'#2ecc71', 0.8:'#3498db', 1.2:'#9b59b6'}

for fv in sorted(all_F_vals):
    if fv == 0.8:
        hists = [r['history'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
    else:
        hists = [r['history'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==fv]
    if hists:
        ml = min(len(h) for h in hists)
        avg_h = np.mean([h[:ml] for h in hists], axis=0)
        ax.plot(safe_log_hist(list(avg_h)), color=colors_F.get(fv,'gray'), linewidth=2, label=f'F={fv}')

ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Average Convergence by F', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='-.', alpha=0.5)

# Cost distribution by F
ax = axes[1]
data_F = []
labels_F = []

for fv in sorted(all_F_vals):
    if fv == 0.8:
        costs = [r['best_cost'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
    else:
        costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==fv]
    data_F.append(safe_log(costs))
    labels_F.append(f'F={fv}')

bp = ax.boxplot(data_F, tick_labels=labels_F, patch_artist=True)
for patch, fv in zip(bp['boxes'], sorted(all_F_vals)):
    patch.set_facecolor(colors_F.get(fv,'gray'))
    patch.set_alpha(0.7)

ax.set_yscale('log')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Cost Distribution by F', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', linestyle='-.', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp3_F_sweep.png', dpi=300, bbox_inches='tight')
plt.show()
print('F sweep boxplot saved.')

Generating F sweep plots...
F sweep boxplot saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/2468861983.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
fig, ax = plt.subplots(figsize=(10, 5))

for fv in sorted(all_F_vals):
    if fv == 0.8:
        hists = [r['history'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
    else:
        hists = [r['history'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==fv]
    if hists:
        ml = min(len(h) for h in hists)
        avg_h = np.mean([h[:ml] for h in hists], axis=0)
        # Zoomed: show last 60% of generations
        start = int(ml * 0.3)
        ax.plot(range(start, ml), safe_log_hist(list(avg_h[start:])), 
                color=colors_F.get(fv,'gray'), linewidth=2, label=f'F={fv}')

ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Zoomed Convergence by F (late generations)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='-.', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp3_F_sweep_zoomed.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/3013039933.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 4: Crossover Rate (CR) Sweep

**Objective:** Determine optimal CR value
**CR values tested:** [0.0, 0.2, 0.5, 0.7] (plus baseline 0.9)
**Strategy:** Best strategy from Exp1/Exp2
**F:** Best F value from Exp3
**Runs per CR:** 5 seeds

In [14]:
CR_values = [0.0, 0.2, 0.5, 0.7]

print('\n' + '='*65)
print(f'EXPERIMENT 4: Varying CR — strategy={best_strategy_12}, F={best_F_val}, ps={PS}')
print('='*65)

for CR_val in CR_values:
    print(f'\n  CR = {CR_val}')
    for seed in seeds:
        run_id += 1
        t0 = time.perf_counter()
        bx, bc, hist, fpop, ffit = run_de(ps=PS, F=best_F_val, CR=CR_val, max_gen=MAX_GEN, 
                                           seed=seed, strategy=best_strategy_12)
        runtime = time.perf_counter() - t0
        results_all.append({
            'run_id': run_id, 'experiment': 'Exp4_CR_sweep',
            'seed': seed, 'strategy': best_strategy_12, 'F': best_F_val, 'CR': CR_val, 
            'ps': PS, 'dynamic_F': False,
            'best_x1': bx[0], 'best_x2': bx[1], 'best_cost': bc,
            'generations': len(hist), 'history': hist, 'runtime_s': runtime,
            'final_pop_mean': float(np.mean(ffit)), 'final_pop_std': float(np.std(ffit))
        })
        print(f'    Seed {seed}: f = {bc:.8f}  at ({bx[0]:.6f}, {bx[1]:.6f})')

# Summary
all_CR_vals = CR_values + [0.9]
best_CR_cost = float('inf')
best_CR_val = 0.9

print('\n--- CR summary ---')
for cv in sorted(all_CR_vals):
    if cv == 0.9:
        if best_F_val == 0.8:
            costs = [r['best_cost'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
        else:
            costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==best_F_val]
    else:
        costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp4_CR_sweep' and r['CR']==cv]
    m = np.mean(costs)
    print(f'  CR={cv}: mean={m:.6f}, std={np.std(costs):.6f}')
    if m < best_CR_cost:
        best_CR_cost = m
        best_CR_val = cv

print(f'\n>>> Best CR: {best_CR_val} (mean={best_CR_cost:.8f})')


EXPERIMENT 4: Varying CR — strategy=rand, F=1.2, ps=10

  CR = 0.0
    Seed 1: f = 0.00000000  at (-0.000000, 0.000000)
    Seed 2: f = 0.00000000  at (0.000000, 0.000000)
    Seed 3: f = 0.00000000  at (0.000000, 0.000000)
    Seed 4: f = 0.00000000  at (0.000000, -0.000000)
    Seed 5: f = 0.00000000  at (0.000000, 0.000000)

  CR = 0.2
    Seed 1: f = 0.00000000  at (-0.000000, -0.000000)
    Seed 2: f = 0.00000000  at (0.000000, -0.000000)
    Seed 3: f = 0.00000000  at (0.000000, -0.000000)
    Seed 4: f = 0.00000000  at (-0.000000, -0.000000)
    Seed 5: f = 0.00000000  at (0.000000, -0.000000)

  CR = 0.5
    Seed 1: f = 0.00000000  at (-0.000000, 0.000000)
    Seed 2: f = 0.00000000  at (-0.000000, -0.000000)
    Seed 3: f = 0.00000000  at (-0.000000, 0.000000)
    Seed 4: f = 0.99495906  at (0.000000, 0.994959)
    Seed 5: f = 0.00000000  at (0.000000, 0.000000)

  CR = 0.7
    Seed 1: f = 0.00000000  at (0.000000, -0.000000)
    Seed 2: f = 0.00000000  at (-0.000000, -0.0000

## CR Sweep Results Visualization

In [15]:
print('Generating CR sweep plots...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence by CR
ax = axes[0]
colors_CR = {0.0:'#e74c3c', 0.2:'#f39c12', 0.5:'#2ecc71', 0.7:'#3498db', 0.9:'#9b59b6'}

for cv in sorted(all_CR_vals):
    if cv == 0.9:
        if best_F_val == 0.8:
            hists = [r['history'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
        else:
            hists = [r['history'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==best_F_val]
    else:
        hists = [r['history'] for r in results_all if r['experiment']=='Exp4_CR_sweep' and r['CR']==cv]
    if hists:
        ml = min(len(h) for h in hists)
        avg_h = np.mean([h[:ml] for h in hists], axis=0)
        ax.plot(safe_log_hist(list(avg_h)), color=colors_CR.get(cv,'gray'), linewidth=2, label=f'CR={cv}')

ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Average Convergence by CR', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='-.', alpha=0.5)

# Cost distribution by CR
ax = axes[1]
data_CR = []
labels_CR = []

for cv in sorted(all_CR_vals):
    if cv == 0.9:
        if best_F_val == 0.8:
            costs = [r['best_cost'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
        else:
            costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==best_F_val]
    else:
        costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp4_CR_sweep' and r['CR']==cv]
    data_CR.append(safe_log(costs))
    labels_CR.append(f'CR={cv}')

bp = ax.boxplot(data_CR, tick_labels=labels_CR, patch_artist=True)
for patch, cv in zip(bp['boxes'], sorted(all_CR_vals)):
    patch.set_facecolor(colors_CR.get(cv,'gray'))
    patch.set_alpha(0.7)

ax.set_yscale('log')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Cost Distribution by CR', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', linestyle='-.', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp4_CR_sweep.png', dpi=300, bbox_inches='tight')
plt.show()
print('CR sweep plots saved.')

Generating CR sweep plots...
CR sweep plots saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/2546842002.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, cv in enumerate(sorted(all_CR_vals)):
    ax = axes[idx]
    if cv == 0.9:
        if best_F_val == 0.8:
            subset = [r for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
        else:
            subset = [r for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==best_F_val]
    else:
        subset = [r for r in results_all if r['experiment']=='Exp4_CR_sweep' and r['CR']==cv]
    
    for r in subset:
        ax.plot(safe_log_hist(r['history']), alpha=0.7, linewidth=1, label=f"seed={int(r['seed'])}")
    
    ax.set_yscale('log')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Best Cost (log)')
    ax.set_title(f'CR = {cv}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='-.', alpha=0.4)

axes[5].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp4_CR_per_seed.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/538821189.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 5: Dynamic F

**Objective:** Compare fixed F vs dynamic F sampled from U(0.5, 1.0)
**Strategy:** Best from Exp1/Exp2
**CR:** Best from Exp4
**Dynamic F:** Sampled from Uniform(0.5, 1.0) each generation
**Runs:** 5 seeds

In [17]:
print('\n' + '='*65)
print(f'EXPERIMENT 5: Dynamic F ~ U(0.5, 1.0) — strategy={best_strategy_12}, CR={best_CR_val}, ps={PS}')
print('='*65)

for seed in seeds:
    run_id += 1
    t0 = time.perf_counter()
    bx, bc, hist, fpop, ffit = run_de(ps=PS, F=0.8, CR=best_CR_val, max_gen=MAX_GEN, seed=seed,
                                       strategy=best_strategy_12, dynamic_F=True, F_low=0.5, F_high=1.0)
    runtime = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'Exp5_dynamic_F',
        'seed': seed, 'strategy': best_strategy_12, 'F': 'U(0.5,1.0)', 'CR': best_CR_val, 
        'ps': PS, 'dynamic_F': True,
        'best_x1': bx[0], 'best_x2': bx[1], 'best_cost': bc,
        'generations': len(hist), 'history': hist, 'runtime_s': runtime,
        'final_pop_mean': float(np.mean(ffit)), 'final_pop_std': float(np.std(ffit))
    })
    print(f'  Seed {seed}: f = {bc:.8f}  at ({bx[0]:.6f}, {bx[1]:.6f})  [{runtime:.3f}s]')

exp5_costs = [r['best_cost'] for r in results_all if r['experiment'] == 'Exp5_dynamic_F']
print(f'\nExp5 summary: mean={np.mean(exp5_costs):.8f}, std={np.std(exp5_costs):.8f}')

print(f'\n{"="*65}')
print(f'TOTAL RUNS: {run_id}  (expected: 55)')
print(f'{"="*65}')


EXPERIMENT 5: Dynamic F ~ U(0.5, 1.0) — strategy=rand, CR=0.0, ps=10
  Seed 1: f = 0.99495906  at (-0.994959, 0.000000)  [0.038s]
  Seed 2: f = 0.00000000  at (-0.000000, 0.000000)  [0.037s]
  Seed 3: f = 0.00000000  at (-0.000000, -0.000000)  [0.037s]
  Seed 4: f = 0.00000000  at (0.000000, -0.000000)  [0.036s]
  Seed 5: f = 0.00000000  at (-0.000000, -0.000000)  [0.036s]

Exp5 summary: mean=0.19899181, std=0.39798362

TOTAL RUNS: 55  (expected: 55)


## Dynamic F Results Visualization

In [18]:
print('Generating Dynamic F plots...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
if best_F_val == 0.8:
    fixed_hists = [r['history'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
else:
    fixed_hists = [r['history'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==best_F_val]

dyn_hists = [r['history'] for r in results_all if r['experiment']=='Exp5_dynamic_F']
ml = min(min(len(h) for h in fixed_hists), min(len(h) for h in dyn_hists))
avg_fixed = np.mean([h[:ml] for h in fixed_hists], axis=0)
avg_dyn = np.mean([h[:ml] for h in dyn_hists], axis=0)

ax.plot(safe_log_hist(list(avg_fixed)), color='navy', linewidth=2, label=f'Fixed F={best_F_val}')
ax.plot(safe_log_hist(list(avg_dyn)), color='darkorange', linewidth=2, label='Dynamic F ~ U(0.5,1.0)')
ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Fixed F vs Dynamic F', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='-.', alpha=0.5)

ax = axes[1]
for r in results_all:
    if r['experiment']=='Exp5_dynamic_F':
        ax.plot(safe_log_hist(r['history']), linewidth=1.2, label=f"seed={int(r['seed'])}")

ax.set_yscale('log')
ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Dynamic F: Per-Seed Convergence', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='-.', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/exp5_dynamic_F.png', dpi=300, bbox_inches='tight')
plt.show()
print('Dynamic F plots saved.')

Generating Dynamic F plots...
Dynamic F plots saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/894116428.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Overall Results Summary

Comprehensive analysis across all 5 experiment types and 55 runs.

In [19]:
# Convert to DataFrame
results_df = pd.DataFrame(results_all)

# Define experiment labels
exp_labels = {
    'Exp1_rand': '1: DE/rand/1/bin',
    'Exp2_best': '2: DE/best/1/bin',
    'Exp3_F_sweep': '3: F Sweep',
    'Exp4_CR_sweep': '4: CR Sweep',
    'Exp5_dynamic_F': '5: Dynamic F'
}

experiments = ['Exp1_rand', 'Exp2_best', 'Exp3_F_sweep', 'Exp4_CR_sweep', 'Exp5_dynamic_F']

print('Results DataFrame shape:', results_df.shape)
print('\nExperiment counts:')
print(results_df['experiment'].value_counts().sort_index())

Results DataFrame shape: (55, 16)

Experiment counts:
experiment
Exp1_rand          5
Exp2_best          5
Exp3_F_sweep      20
Exp4_CR_sweep     20
Exp5_dynamic_F     5
Name: count, dtype: int64


In [20]:
print('Generating overall boxplot...')

fig, ax = plt.subplots(figsize=(10, 6))
data = [safe_log(list(results_df[results_df['experiment']==e]['best_cost'].values)) for e in experiments]
labels = [exp_labels[e] for e in experiments]

bp = ax.boxplot(data, tick_labels=labels, patch_artist=True)
colors = ['#3498db','#e74c3c','#2ecc71','#9b59b6','#f39c12']

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_yscale('log')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Best Cost by DE Experiment', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', linestyle='-.', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/boxplot_by_experiment.png', dpi=400, bbox_inches='tight')
plt.show()
print('Overall boxplot saved.')

Generating overall boxplot...
Overall boxplot saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/2683899.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, exp in enumerate(experiments):
    ax = axes[idx]
    subset = [r for r in results_all if r['experiment']==exp]
    for r in subset:
        ax.plot(safe_log_hist(r['history']), alpha=0.5, linewidth=0.8)
    
    ax.set_yscale('log')
    ax.set_xlabel('Generation', fontsize=11)
    ax.set_ylabel('Best Cost (log)', fontsize=11)
    ax.set_title(exp_labels[exp], fontsize=13, fontweight='bold')
    ax.grid(True, linestyle='-.', alpha=0.4)

axes[5].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTDIR}/convergence_all_experiments.png', dpi=300, bbox_inches='tight')
plt.show()
print('Convergence all experiments plot saved.')

Convergence all experiments plot saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/931175084.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
best_overall = results_df.loc[results_df['best_cost'].idxmin()]

print(f'Overall best solution:')
print(f'  Experiment: {best_overall["experiment"]}')
print(f'  Seed: {best_overall["seed"]}')
print(f'  F: {best_overall["F"]}, CR: {best_overall["CR"]}')
print(f'  Best cost: {best_overall["best_cost"]:.10f}')
print(f'  Best x: ({best_overall["best_x1"]:.10f}, {best_overall["best_x2"]:.10f})')

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(X1, X2, Z, levels=50, cmap='viridis')
plt.colorbar(c, ax=ax)
ax.plot(0, 0, 'r*', markersize=20, label='Global min (0,0)', zorder=5)
ax.plot(best_overall['best_x1'], best_overall['best_x2'], 'wo', markersize=14,
        markeredgecolor='red', markeredgewidth=2.5, zorder=5,
        label=f"Best DE ({best_overall['best_x1']:.6f}, {best_overall['best_x2']:.6f})")
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title(f"Best DE Solution  |  f = {best_overall['best_cost']:.8f}", fontsize=13)
ax.legend(loc='upper right')
plt.savefig(f'{OUTDIR}/best_solution_contour.png', dpi=150, bbox_inches='tight')
plt.show()
print('Best solution contour plot saved.')

Overall best solution:
  Experiment: Exp1_rand
  Seed: 2
  F: 0.8, CR: 0.9
  Best cost: 0.0000000000
  Best x: (-0.0000000012, 0.0000000016)
Best solution contour plot saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/4221184380.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
fig, ax = plt.subplots(figsize=(8, 5))

colors_map = {
    'Exp1_rand':'#3498db', 
    'Exp2_best':'#e74c3c', 
    'Exp3_F_sweep':'#2ecc71',
    'Exp4_CR_sweep':'#9b59b6', 
    'Exp5_dynamic_F':'#f39c12'
}

for exp in experiments:
    sub = results_df[results_df['experiment']==exp]
    costs_safe = [max(v, 1e-16) for v in sub['best_cost'].values]
    ax.scatter(sub['runtime_s'], costs_safe, c=colors_map[exp],
               label=exp_labels[exp], s=80, edgecolors='#2c3e50', linewidth=0.5, alpha=0.8)

ax.set_yscale('log')
ax.set_xlabel('Runtime (seconds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (log scale)', fontsize=12, fontweight='bold')
ax.set_title('Runtime vs Cost Efficiency', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='both', linestyle='-.')
plt.tight_layout()
plt.savefig(f'{OUTDIR}/runtime_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print('Runtime vs cost plot saved.')

Runtime vs cost plot saved.


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_39075/624137876.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Statistical Summary

In [24]:
summary = pd.DataFrame({
    'Experiment': [exp_labels[e] for e in experiments],
    'Num Runs': [len(results_df[results_df['experiment']==e]) for e in experiments],
    'Mean Cost': [results_df[results_df['experiment']==e]['best_cost'].mean() for e in experiments],
    'Std': [results_df[results_df['experiment']==e]['best_cost'].std() for e in experiments],
    'Min Cost': [results_df[results_df['experiment']==e]['best_cost'].min() for e in experiments],
    'Max Cost': [results_df[results_df['experiment']==e]['best_cost'].max() for e in experiments],
    'Mean Runtime (s)': [results_df[results_df['experiment']==e]['runtime_s'].mean() for e in experiments],
})

print('\n' + '='*90)
print('SUMMARY TABLE')
print('='*90)
print(summary.to_string(index=False))

summary.to_csv(f'{OUTDIR}/summary_table.csv', index=False)
print(f'\nSummary saved to {OUTDIR}/summary_table.csv')


SUMMARY TABLE
      Experiment  Num Runs  Mean Cost      Std  Min Cost  Max Cost  Mean Runtime (s)
1: DE/rand/1/bin         5   0.596975 0.544962  0.000000  0.994959          0.040137
2: DE/best/1/bin         5   2.217993 1.841730  0.053516  4.974790          0.036076
      3: F Sweep        20   1.379204 2.204314  0.000000  8.471419          0.033736
     4: CR Sweep        20   0.049748 0.222480  0.000000  0.994959          0.036390
    5: Dynamic F         5   0.198992 0.444959  0.000000  0.994959          0.036759

Summary saved to ./DE_Results/summary_table.csv


## Export Results to CSV and Excel

In [25]:
print('Saving results...')

# Save CSV without history column
save_cols = [c for c in results_df.columns if c != 'history']
results_df[save_cols].to_csv(f'{OUTDIR}/all_55_runs.csv', index=False)
print(f'Saved: {OUTDIR}/all_55_runs.csv')

# Save histories as JSON
histories = {}
for r in results_all:
    key = f"{r['experiment']}_seed{r['seed']}_F{r['F']}_CR{r['CR']}"
    histories[key] = r['history']

with open(f'{OUTDIR}/histories.json', 'w') as f:
    json.dump(histories, f)
print(f'Saved: {OUTDIR}/histories.json')

# Try to save Excel
try:
    results_df[save_cols].to_excel(f'{OUTDIR}/All_Experiments.xlsx', index=False)
    print(f'Saved: {OUTDIR}/All_Experiments.xlsx')
except:
    print('Excel export skipped (openpyxl not available)')

print('\nAll results saved successfully!')

Saving results...
Saved: ./DE_Results/all_55_runs.csv
Saved: ./DE_Results/histories.json
Saved: ./DE_Results/All_Experiments.xlsx

All results saved successfully!


## Key Findings and Conclusions

In [26]:
print('\n' + '='*70)
print('KEY RESULTS FOR REPORT')
print('='*70)
print(f'ps={PS}, max_gen={MAX_GEN}')
print(f'Best strategy: {best_strategy_12}')
print(f'Best F: {best_F_val}')
print(f'Best CR: {best_CR_val}')

best = results_df.loc[results_df['best_cost'].idxmin()]
print(f'\nOverall best solution:')
print(f'  Experiment={best["experiment"]}, seed={int(best["seed"])}, F={best["F"]}, CR={best["CR"]}')
print(f'  f* = {best["best_cost"]:.10f}')
print(f'  x* = ({best["best_x1"]:.10f}, {best["best_x2"]:.10f})')

print(f'\n--- Exp1 Per-Seed ---')
for r in results_all:
    if r['experiment'] == 'Exp1_rand':
        print(f"  Seed {r['seed']}: cost={r['best_cost']:.8f}, pop_mean={r['final_pop_mean']:.8f}, pop_std={r['final_pop_std']:.8f}")

print(f'\n--- Exp2 Per-Seed ---')
for r in results_all:
    if r['experiment'] == 'Exp2_best':
        print(f"  Seed {r['seed']}: cost={r['best_cost']:.8f}, pop_mean={r['final_pop_mean']:.8f}, pop_std={r['final_pop_std']:.8f}")

print(f'\n--- F Value Table ---')
for fv in sorted(all_F_vals):
    if fv == 0.8:
        costs = [r['best_cost'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
    else:
        costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==fv]
    print(f"F={fv}: mean={np.mean(costs):.6f}, std={np.std(costs):.6f}")

print(f'\n--- CR Value Table ---')
for cv in sorted(all_CR_vals):
    if cv == 0.9:
        if best_F_val == 0.8:
            costs = [r['best_cost'] for r in results_all if r['experiment'] in ('Exp1_rand','Exp2_best') and r['strategy']==best_strategy_12]
        else:
            costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp3_F_sweep' and r['F']==best_F_val]
    else:
        costs = [r['best_cost'] for r in results_all if r['experiment']=='Exp4_CR_sweep' and r['CR']==cv]
    print(f"CR={cv}: mean={np.mean(costs):.6f}, std={np.std(costs):.6f}")

print(f'\n--- Exp5 Per-Seed (Dynamic F) ---')
for r in results_all:
    if r['experiment'] == 'Exp5_dynamic_F':
        print(f"  Seed {r['seed']}: cost={r['best_cost']:.8f}, pop_mean={r['final_pop_mean']:.8f}, pop_std={r['final_pop_std']:.8f}")

print('\n' + '='*70)
print('NOTEBOOK EXECUTION COMPLETE')
print('='*70)


KEY RESULTS FOR REPORT
ps=10, max_gen=500
Best strategy: rand
Best F: 1.2
Best CR: 0.0

Overall best solution:
  Experiment=Exp1_rand, seed=2, F=0.8, CR=0.9
  f* = 0.0000000000
  x* = (-0.0000000012, 0.0000000016)

--- Exp1 Per-Seed ---
  Seed 1: cost=0.99495906, pop_mean=0.99495906, pop_std=0.00000000
  Seed 2: cost=0.00000000, pop_mean=0.00000000, pop_std=0.00000000
  Seed 3: cost=0.99495906, pop_mean=0.99495906, pop_std=0.00000000
  Seed 4: cost=0.00000000, pop_mean=0.00000000, pop_std=0.00000000
  Seed 5: cost=0.99495906, pop_mean=0.99495906, pop_std=0.00000000

--- Exp2 Per-Seed ---
  Seed 1: cost=2.19019741, pop_mean=2.19019741, pop_std=0.00000000
  Seed 2: cost=0.05351581, pop_mean=0.05351581, pop_std=0.00000000
  Seed 3: cost=2.68502014, pop_mean=2.68502014, pop_std=0.00000000
  Seed 4: cost=4.97479025, pop_mean=4.97479025, pop_std=0.00000000
  Seed 5: cost=1.18644192, pop_mean=1.18644192, pop_std=0.00000000

--- F Value Table ---
F=0.1: mean=4.093879, std=2.715079
F=0.3: mean